In [1]:
import pandas as pd
import sqlite3
import os

# 현재 작업 경로 확인
print("현재 작업 폴더:", os.getcwd())

# DB 연결
conn = sqlite3.connect("../db/olist.db")

# customers 데이터 불러오기
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

# orders 데이터 불러오기
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

# DB에 저장
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

# 테이블 확인
print(pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn))

# customers 미리보기
pd.read_sql("""
SELECT *
FROM customers
LIMIT 5;
""", conn)

현재 작업 폴더: /Users/chaeyoung/Documents/GitHub/olist-ecommerce-analysis/notebooks
                                name
0                          customers
1                        geolocation
2                        order_items
3                     order_payments
4                      order_reviews
5                             orders
6  product_category_name_translation
7                           products
8                            sellers
9                         test_table


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [2]:
# -- 고객 기본 현황 분석

# 전체 고객 수
display(pd.read_sql("""
SELECT COUNT(*) AS total_customers
FROM customers;
""", conn))

# 고유 고객 수(customer_unique_id 기준)
display(pd.read_sql("""
SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM customers;
""", conn))

# 주(state)별 고객 수
display(pd.read_sql("""
SELECT
    customer_state,
    COUNT(*) AS customer_count
FROM customers
GROUP BY customer_state
ORDER BY customer_count DESC;
""", conn))

# 도시(city)별 고객 수 상위 20개
display(pd.read_sql("""
SELECT
    customer_city,
    COUNT(*) AS customer_count
FROM customers
GROUP BY customer_city
ORDER BY customer_count DESC
LIMIT 20;
""", conn))
# 전체 고객 수
display(pd.read_sql("""
SELECT COUNT(*) AS total_customers
FROM customers;
""", conn))

# 고유 고객 수(customer_unique_id 기준)
display(pd.read_sql("""
SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM customers;
""", conn))

# 주(state)별 고객 수
display(pd.read_sql("""
SELECT
    customer_state,
    COUNT(*) AS customer_count
FROM customers
GROUP BY customer_state
ORDER BY customer_count DESC;
""", conn))

# 도시(city)별 고객 수 상위 20개
display(pd.read_sql("""
SELECT
    customer_city,
    COUNT(*) AS customer_count
FROM customers
GROUP BY customer_city
ORDER BY customer_count DESC
LIMIT 20;
""", conn))

,total_customers
0,99441


,unique_customers
0,96096


,customer_state,customer_count
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


,customer_city,customer_count
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


,total_customers
0,99441


,unique_customers
0,96096


,customer_state,customer_count
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


,customer_city,customer_count
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


In [3]:
#-- 셀 3. 고객 주문 분석

# 고객 1인당 주문 수 분포
display(pd.read_sql("""
SELECT
    order_count,
    COUNT(*) AS customer_count
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS order_count
    FROM customers c
    LEFT JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
GROUP BY order_count
ORDER BY order_count;
""", conn))

# 고객 1인당 평균 주문 수
display(pd.read_sql("""
SELECT
    ROUND(AVG(order_count), 2) AS avg_orders_per_customer
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS order_count
    FROM customers c
    LEFT JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
);
""", conn))

# 1회 구매 고객 수와 비율
display(pd.read_sql("""
SELECT
    COUNT(*) AS one_time_customers,
    ROUND(COUNT(*) * 100.0 / (
        SELECT COUNT(DISTINCT customer_unique_id)
        FROM customers
    ), 2) AS one_time_customer_pct
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS order_count
    FROM customers c
    LEFT JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
    HAVING COUNT(o.order_id) = 1
);
""", conn))

# 재구매 고객 수와 비율
display(pd.read_sql("""
SELECT
    COUNT(*) AS repeat_customers,
    ROUND(COUNT(*) * 100.0 / (
        SELECT COUNT(DISTINCT customer_unique_id)
        FROM customers
    ), 2) AS repeat_customer_pct
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS order_count
    FROM customers c
    LEFT JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
    HAVING COUNT(o.order_id) >= 2
);
""", conn))

,order_count,customer_count
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


,avg_orders_per_customer
0,1.03


,one_time_customers,one_time_customer_pct
0,93099,96.88


,repeat_customers,repeat_customer_pct
0,2997,3.12


In [4]:
#--셀 4. 고객 지역 + 주문 결합 분석

# 주(state)별 주문 수
display(pd.read_sql("""
SELECT
    c.customer_state,
    COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_state
ORDER BY order_count DESC;
""", conn))

# 주(state)별 고유 고객 수 대비 주문 수
display(pd.read_sql("""
SELECT
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
    COUNT(o.order_id) AS total_orders,
    ROUND(
        COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_unique_id),
        2
    ) AS avg_orders_per_customer
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC;
""", conn))

# 주문 수가 많은 고객 상위 20명
display(pd.read_sql("""
SELECT
    c.customer_unique_id,
    COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
ORDER BY order_count DESC
LIMIT 20;
""", conn))

,customer_state,order_count
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


,customer_state,unique_customers,total_orders,avg_orders_per_customer
0,SP,40302,41746,1.04
1,RJ,12384,12852,1.04
2,MG,11259,11635,1.03
3,RS,5277,5466,1.04
4,PR,4882,5045,1.03
5,SC,3534,3637,1.03
6,BA,3277,3380,1.03
7,DF,2075,2140,1.03
8,ES,1964,2033,1.04
9,GO,1952,2020,1.03


,customer_unique_id,order_count
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
